In [ ]:
import os
import sys

# Databricks does not automatically resolve sibling folders.
# Add both this notebook's directory and its parent, so `utils` resolves
# regardless of how the workspace reports the working directory.
for candidate in (os.getcwd(), os.path.dirname(os.getcwd())):
    if candidate not in sys.path:
        sys.path.append(candidate)

from utils.transformations import Reusable

bronze = "abfss://bronze@glcazurestorageproject.dfs.core.windows.net"
silver = "abfss://silver@glcazurestorageproject.dfs.core.windows.net"

print("Import OK:", Reusable)

In [ ]:
df_user = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{silver}/DimUser/schema")
    .load(f"{bronze}/DimUser"))

df_user = Reusable.uppercase(df_user, "user_name")
df_user = Reusable.drop_columns(df_user, "_rescued_data")

(df_user.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{silver}/DimUser/checkpoint")
    .option("path", f"{silver}/DimUser/data")
    .trigger(availableNow=True)
    .toTable("glc_project.silver.dim_user"))

In [ ]:
df_track = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{silver}/DimTrack/schema")
    .load(f"{bronze}/DimTrack"))

df_track = Reusable.bucket_numeric(df_track, "duration_sec", "duration_flag", 150, 300)
df_track = Reusable.drop_columns(df_track, "_rescued_data")

(df_track.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{silver}/DimTrack/checkpoint")
    .option("path", f"{silver}/DimTrack/data")
    .trigger(availableNow=True)
    .toTable("glc_project.silver.dim_track"))

In [ ]:
df_artist = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{silver}/DimArtist/schema")
    .load(f"{bronze}/DimArtist"))

df_artist = Reusable.drop_columns(df_artist, "_rescued_data")

(df_artist.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{silver}/DimArtist/checkpoint")
    .option("path", f"{silver}/DimArtist/data")
    .trigger(availableNow=True)
    .toTable("glc_project.silver.dim_artist"))

In [ ]:
df_date = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{silver}/DimDate/schema")
    .load(f"{bronze}/DimDate"))

df_date = Reusable.drop_columns(df_date, "_rescued_data")

(df_date.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{silver}/DimDate/checkpoint")
    .option("path", f"{silver}/DimDate/data")
    .trigger(availableNow=True)
    .toTable("glc_project.silver.dim_date"))

In [ ]:
df_fact = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{silver}/FactStream/schema")
    .load(f"{bronze}/FactStream"))

df_fact = Reusable.drop_columns(df_fact, "_rescued_data")

(df_fact.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{silver}/FactStream/checkpoint")
    .option("path", f"{silver}/FactStream/data")
    .trigger(availableNow=True)
    .toTable("glc_project.silver.fact_stream"))

In [ ]:
%sql
SELECT 'dim_user' AS tbl, COUNT(*) AS rows FROM glc_project.silver.dim_user
UNION ALL SELECT 'dim_track', COUNT(*) FROM glc_project.silver.dim_track
UNION ALL SELECT 'dim_artist', COUNT(*) FROM glc_project.silver.dim_artist
UNION ALL SELECT 'dim_date', COUNT(*) FROM glc_project.silver.dim_date
UNION ALL SELECT 'fact_stream', COUNT(*) FROM glc_project.silver.fact_stream;

In [ ]:
%sql
SELECT COUNT(*) FROM glc_project.silver.dim_user

In [ ]:
%sql
SELECT COUNT(*) FROM parquet.`abfss://bronze@glcazurestorageproject.dfs.core.windows.net/DimUser`

In [ ]:
%sql
SELECT * FROM glc_project.silver.dim_user LIMIT 10